# Task 2: Sentiment & Thematic Analysis

This notebook documents methodology for sentiment classification and theme grouping across Ethiopian bank app reviews.

## Tool Selection Rationale

| Tool | Role | Why |
|------|------|-----|
| **DistilBERT SST-2** | Primary sentiment | Fine-tuned transformer; better on context and informal phrasing than lexicons |
| **VADER** | Comparison baseline | Fast, interpretable; useful to sanity-check transformer output |
| **NLTK + TF-IDF** | Text prep & keywords | Lightweight, reproducible; extracts n-grams per bank for theme validation |

Neutral labels are assigned when DistilBERT confidence &lt; 0.55 (low certainty on positive vs negative).

## Theme Grouping Logic

We define **5 business-relevant themes** with keyword/n-gram triggers:

1. **Account Access Issues** — login, password, OTP, verification, locked account
2. **Transaction Performance** — transfer, slow, pending, payment failed, delay
3. **UI & Design** — interface, layout, easy to use, navigation, screen
4. **Customer Support** — support, branch, staff, help, complaint, response time
5. **Feature Requests** — add feature, update, missing, should include

**Assignment rule:** Each review is scored by counting keyword hits per theme; the theme with the highest hit count wins. TF-IDF top terms per bank are used to validate that extracted keywords align with dominant discourse (see `analysis_stats.json`).

Reviews with no keyword hits → **General Feedback**.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd() / "src").exists() is False else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

stats = json.loads((ROOT / "data/processed/analysis_stats.json").read_text())
analyzed = pd.read_csv(ROOT / "data/processed/reviews_analyzed.csv")
print(f"Reviews analyzed: {len(analyzed)}")
print(f"Sentiment labeled: {stats['sentiment_labeled_pct']}%")
print("\nSentiment distribution:", stats["sentiment_distribution"])

In [ ]:
agg = pd.DataFrame(stats["aggregate_by_bank_rating"])
agg.pivot_table(index="bank", columns="rating", values="mean_sentiment_score")

In [ ]:
for bank, info in stats["bank_theme_summary"].items():
    print(f"\n{bank}")
    print("  Theme counts:", info["theme_counts"])
    print("  TF-IDF terms:", info["tfidf_top_terms"][:8])